In [4]:
"""Data loader for the Telugu dataset."""
import os
import re
import sys
import codecs
import unicodedata
import numpy as np

from torch.utils.data import Dataset

vocab_telugu = "PE అఆఇఈఉఊఋౠఎఏఐఒఓఔకఖగఘఙచఛజఝఞటఠడఢణతథదధనపఫబభమయరలవశషసహ'.?"  # P: Padding, E: EOS.
char2idx_telugu = {char: idx for idx, char in enumerate(vocab_telugu)}
idx2char_telugu = {idx: char for idx, char in enumerate(vocab_telugu)}


def text_normalize_telugu(text):
    text = ''.join(char for char in unicodedata.normalize('NFD', text)
                   if unicodedata.category(char) != 'Mn')  # Strip accents

    text = text.lower()
    text = re.sub("[^{}]".format(vocab_telugu), " ", text)
    text = re.sub("[ ]+", " ", text)
    return text


def read_metadata_telugu(metadata_file):
    fnames, texts = [], []
    transcript = os.path.join(metadata_file)
    lines = codecs.open(transcript, 'r', 'utf-8').readlines()
    for line in lines:

        # Split the line into two values
        line_values = line.strip().split("|")

        if len(line_values) >= 2:
            fname, text = line_values[:2]  # Take the first two values
            fnames.append(fname)

            text = text_normalize_telugu(text) + "E"  # E: EOS
            text = [char2idx_telugu[char] for char in text]
            texts.append(np.array(text, np.long))

    return fnames, texts

class TeluguDataset(Dataset):
    def __init__(self, keys, dir_name='te_in_male'):
        self.keys = keys
        self.vocab = vocab_telugu
        self.path = os.path.join(os.path.dirname(os.path.realpath(__file__)), dir_name)
        self.fnames, self.texts = read_metadata_telugu(os.path.join(self.path, 'te_in_male.tsv'))

    def slice(self, start, end):
        self.fnames = self.fnames[start:end]
        self.texts = self.texts[start:end]

    def __len__(self):
        return len(self.fnames)

    def __getitem__(self, index):
        data = {}

        if 'texts' in self.keys:
            data['texts'] = self.texts[index]

        if 'mels' in self.keys:
            # Load or preprocess mel data
            mel_fname = os.path.join(self.path, 'mels', "%s.npy" % self.fnames[index])
            mel_data = np.load(mel_fname)
            data['mels'] = mel_data

        if 'mags' in self.keys:
            # Load or preprocess mag data
            mag_fname = os.path.join(self.path, 'mags', "%s.npy" % self.fnames[index])
            mag_data = np.load(mag_fname)
            data['mags'] = mag_data

        if 'mel_gates' in self.keys:
            data['mel_gates'] = np.ones(data.get('mels', np.array([])).shape[0], dtype=np.int)

        if 'mag_gates' in self.keys:
            data['mag_gates'] = np.ones(data.get('mags', np.array([])).shape[0], dtype=np.int)

        return data
